# 🛒 Notebook 01: Exploratory Data Analysis (EDA)
## Retail Customer Purchase Prediction & Behavioral Analysis

**Author:** AI/ML Engineer  
**Objective:** Perform a rigorous exploratory data analysis of the 1,000,000-row transaction dataset to uncover purchasing patterns, temporal trends, basket sizes, demographic distributions, and customer repeat purchase dynamics.


In [ ]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# Add project root to sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.data.loader import load_transactions

# Configure plotting style
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["font.size"] = 10


### 1. Ingest Data via Cached Parquet Pipeline
We leverage the project's Parquet caching loader to quickly load the cleaned transactions.


In [ ]:
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "Retail_Transactions_Dataset.csv"

# Load transactions (uses cached Parquet if generated)
transactions, quality = load_transactions(DATA_PATH, sample_frac=None, use_cache=True)
print(f"Total Transactions: {quality['valid_rows']:,}")
print(f"Unique Customers:   {quality['unique_customers']:,}")
print(f"Date Range:         {quality['date_start']} to {quality['date_end']}")
transactions.head()


### 2. Transaction Cost and Basket Size Distributions
Analyzing spending distribution, order value outliers, and items per basket.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Total Cost distribution
sns.histplot(transactions["Total_Cost"], bins=40, kde=True, ax=axes[0], color="#2b5c8f")
axes[0].set_title("Distribution of Transaction Total Cost ($)")
axes[0].set_xlabel("Total Cost ($)")
axes[0].axvline(transactions["Total_Cost"].mean(), color="red", linestyle="--", label=f"Mean: ${transactions['Total_Cost'].mean():.2f}")
axes[0].axvline(transactions["Total_Cost"].median(), color="orange", linestyle=":", label=f"Median: ${transactions['Total_Cost'].median():.2f}")
axes[0].legend()

# Total Items distribution
sns.countplot(data=transactions, x="Total_Items", ax=axes[1], palette="Blues_r")
axes[1].set_title("Distribution of Basket Item Counts")
axes[1].set_xlabel("Total Items in Basket")

plt.tight_layout()
plt.show()


### 3. Temporal & Seasonality Analysis
Tracking monthly revenue growth, order volume trends, and seasonality.


In [ ]:
monthly_rev = transactions.groupby("Transaction_Month")["Total_Cost"].agg(["sum", "count"]).reset_index()

fig, ax1 = plt.subplots(figsize=(13, 5))
ax2 = ax1.twinx()

ax1.plot(monthly_rev["Transaction_Month"], monthly_rev["sum"] / 1e6, color="#1f77b4", marker="o", linewidth=2, label="Revenue ($M)")
ax2.bar(monthly_rev["Transaction_Month"], monthly_rev["count"], alpha=0.25, color="gray", label="Transaction Volume")

ax1.set_xlabel("Month")
ax1.set_ylabel("Revenue ($ Millions)", color="#1f77b4")
ax2.set_ylabel("Transaction Count", color="gray")
ax1.set_title("Monthly Revenue & Transaction Volume Trajectory")
ax1.tick_params(axis="x", rotation=45)
plt.show()


### 4. Categorical Breakdowns: Demographics, Store Types, & Payment Channels


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# By City
city_rev = transactions.groupby("City")["Total_Cost"].sum().sort_values(ascending=False)
city_rev.plot(kind="bar", ax=axes[0, 0], color="#3470a3")
axes[0, 0].set_title("Total Revenue by City")
axes[0, 0].tick_params(axis="x", rotation=30)

# By Store Type
store_rev = transactions.groupby("Store_Type")["Total_Cost"].sum().sort_values(ascending=False)
store_rev.plot(kind="bar", ax=axes[0, 1], color="#4990c2")
axes[0, 1].set_title("Total Revenue by Store Type")
axes[0, 1].tick_params(axis="x", rotation=30)

# By Customer Category
cat_rev = transactions.groupby("Customer_Category")["Total_Cost"].sum().sort_values(ascending=False)
cat_rev.plot(kind="bar", ax=axes[1, 0], color="#6aaed6")
axes[1, 0].set_title("Revenue by Customer Demographics")
axes[1, 0].tick_params(axis="x", rotation=30)

# By Payment Method
pay_counts = transactions["Payment_Method"].value_counts()
pay_counts.plot(kind="pie", ax=axes[1, 1], autopct="%1.1f%%", colors=["#1f77b4", "#aec7e8", "#ffbb78", "#2ca02c"])
axes[1, 1].set_ylabel("")
axes[1, 1].set_title("Payment Method Share")

plt.tight_layout()
plt.show()


### 5. Customer Purchase Frequency & Repeat Purchase Analysis
Quantifying customer retention: how many customers return for a second or third order?


In [ ]:
order_counts = transactions.groupby("Customer_Name").size()
repeat_customers = (order_counts > 1).sum()
single_customers = (order_counts == 1).sum()

print(f"Total Unique Customers:         {len(order_counts):,}")
print(f"Customers with 1 Transaction:   {single_customers:,} ({single_customers/len(order_counts):.1%})")
print(f"Repeat Customers (>= 2 Orders): {repeat_customers:,} ({repeat_customers/len(order_counts):.1%})")

plt.figure(figsize=(8, 4))
sns.histplot(order_counts[order_counts <= 15], bins=15, discrete=True, color="#2b5c8f")
plt.title("Distribution of Lifetime Purchase Frequency per Customer")
plt.xlabel("Number of Recorded Transactions")
plt.ylabel("Number of Customers")
plt.show()


### 6. Key Business & Modeling Takeaways
1. **Repeat Purchase Sparsity:** A significant percentage of customers have single purchases, underscoring the importance of class-balanced loss functions and ROC-AUC / Average Precision evaluation.
2. **Predictive Features:** Monetary spend, frequency counts, customer demographic category, and city store contexts provide strong variance signals for repeat purchase likelihood.
3. **Temporal Stability:** Transaction counts remain distributed across months without erratic spikes, making chronological time splits viable.
